# 在Agent写代码前，先把任务框出来

一个编码agent可以快速实现一次清晰的任务。同样也可以快速实现一次模糊的任务。速度是相同的，代价却不同。

Agent 很会填空，你的工作是先决定哪些空格不许它瞎填。

## 昂贵的失败

“给邮件添加重复保护”听起来很具体，并不是。在哪一层实现去重逻辑、是否区分大小写...

有能力的agent会用看似合理的选择填补这些空白。这恰恰危险：实现可以干净、测过、但是与系统却不兼容。

因此，编码agent工作的第一项工组不是编辑代码，而是制定仓库证据支撑的任务框架。

## 任务框架

有用的任务框架包含六个字段：
目标、仓库事实、允许路径、禁止路径、验收证据、未知项处理。

```mermaid
flowchart LR
A[请求]-->B[检查仓库]-->C[有证据支撑的事实]-->D[边界明确的任务框架]-->E[未知项是否阻碍设计]--是-->F[询问或调查]
E--否-->G[变更]
```

## 侦查是在寻找约束

你不用读完整个仓库，而是搜索会约束这次改动的地方。
1. 现在的行为是谁调用的
2. 最接近的现有测试
3. 对外契约（接口返回哪些东西）
4. 项目里的规矩和说明
5. 怎么构建、怎么验证
6. 以前的类似改动是怎么做的

每个计划中的决定，要么有证据、要么明确授权、要么已经写进Unknowns。违反这个限制继续往下，多半是在浪费时间。

### 事实VS假设

没有证据的句子，一律当假设。

### 未知VS假设

未知承认不知道，假设则替未知偷偷填了一个答案。

未知是可控缺口，而假设是埋雷。

## 未知不是失败

可以分成四类：
1. 能查出来的。 翻代码、跑系统能回答
2. 可以自己定的。 用户选或者授权agent选
3. 必须问人的。 可能影响产品行为、成本和兼容性的
4. 这次不做的。 标成非目标，别塞进本轮。

Agent 可以继续啃前两类，碰到必须问人的，要停，别把重大决定埋进代码里。

## 先写验收，在做实现

动手前先写怎么证明做完了。

# 开始编码

对应本章核心：**六字段任务框**、**侦查找约束（不是读完整仓库）**、**事实 / 假设 / 未知**、**must_ask 挡住实现**、**先写验收再改代码**。  
玩具先跑通门禁；生产段用 **LangChain 工具 + DeepSeek** 强制先框。不硬凑 PyTorch。


## 1. 教学玩具：Recon + TaskFrame + FramingGate

- 六类侦查：调用方、最近测试、对外契约、规矩、验证命令、类似改动。
- `start_impl` 在缺验收或未关闭 `must_ask` 时抛错。


In [ ]:
from __future__ import annotations

import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal


UnknownKind = Literal["discoverable", "decidable", "must_ask", "out_of_scope"]


class FrameIncomplete(RuntimeError):
    """任务框未闭合：不许开改。"""


class MustAskBlocking(RuntimeError):
    """必须问人的未知还开着：不许把决定埋进代码。"""


@dataclass
class Fact:
    """有仓库证据的句子。"""

    text: str
    evidence: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class Unknown:
    """可控缺口；不是偷偷填好的假设。"""

    text: str
    kind: UnknownKind
    resolved: bool = False
    resolution: str = ""

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class TaskFrame:
    """六字段任务框。"""

    goal: str
    repo_facts: list[Fact] = field(default_factory=list)
    allowed_paths: list[str] = field(default_factory=list)
    forbidden_paths: list[str] = field(default_factory=list)
    acceptance: list[str] = field(default_factory=list)
    unknowns: list[Unknown] = field(default_factory=list)
    assumptions: list[str] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal": self.goal,
            "repo_facts": [f.to_dict() for f in self.repo_facts],
            "allowed_paths": self.allowed_paths,
            "forbidden_paths": self.forbidden_paths,
            "acceptance": self.acceptance,
            "unknowns": [u.to_dict() for u in self.unknowns],
            "assumptions": self.assumptions,
        }

    def open_must_ask(self) -> list[Unknown]:
        return [u for u in self.unknowns if u.kind == "must_ask" and not u.resolved]

    def validate(self) -> None:
        """
        Raises:
            FrameIncomplete: 缺字段。
            MustAskBlocking: 仍有必须问人的未知。
        """
        if not self.goal.strip():
            raise FrameIncomplete("goal required")
        if not self.repo_facts:
            raise FrameIncomplete("repo_facts required (evidence-backed)")
        if not self.allowed_paths or not self.forbidden_paths:
            raise FrameIncomplete("allowed_paths and forbidden_paths required")
        if not self.acceptance:
            raise FrameIncomplete("write acceptance before implementation")
        if self.open_must_ask():
            raise MustAskBlocking("must_ask unknowns still open; stop and ask")


def classify_sentence(text: str, *, evidence: str | None = None) -> Literal["fact", "assumption", "unknown"]:
    """
    没有证据的句子一律当假设；带问号/未知标记则是未知。

    Args:
        text: 句子。
        evidence: 仓库路径或摘录。
    Returns:
        kind: fact / assumption / unknown。
    """
    t = text.strip()
    if t.endswith("？") or t.endswith("?") or t.lower().startswith("unknown:"):
        return "unknown"
    if evidence:
        return "fact"
    return "assumption"


class Recon:
    """侦查：只搜会约束这次改动的地方，不读完整仓库。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)

    def callers_of(self, symbol: str) -> list[Fact]:
        facts: list[Fact] = []
        for p in self.root.rglob("*.py"):
            body = p.read_text(encoding="utf-8")
            if symbol in body and p.name != "dedupe.py":
                rel = str(p.relative_to(self.root))
                facts.append(Fact(f"{symbol} 被 {rel} 调用", evidence=rel))
        return facts

    def nearest_test(self, topic: str) -> list[Fact]:
        tests = list((self.root / "tests").glob("test_*.py")) if (self.root / "tests").exists() else []
        hits = [p for p in tests if topic in p.read_text(encoding="utf-8") or topic in p.name]
        if not hits:
            return []
        rel = str(hits[0].relative_to(self.root))
        return [Fact(f"最近测试在 {rel}", evidence=rel)]

    def public_contract(self) -> list[Fact]:
        readme = self.root / "README.md"
        if not readme.exists():
            return []
        text = readme.read_text(encoding="utf-8")
        if "idempotent" in text.lower() or "重复" in text:
            return [Fact("对外契约提到重复/幂等", evidence="README.md")]
        return [Fact("README 未声明去重契约", evidence="README.md")]

    def conventions(self) -> list[Fact]:
        agents = self.root / "AGENTS.md"
        if agents.exists() and "src/email" in agents.read_text(encoding="utf-8"):
            return [Fact("路由指向 src/email 为邮件改动入口", evidence="AGENTS.md")]
        return []

    def verify_command(self) -> list[Fact]:
        wb = self.root / "workbench.json"
        if not wb.exists():
            return []
        cmd = json.loads(wb.read_text(encoding="utf-8")).get("test_command", "")
        if cmd:
            return [Fact(f"验证命令是 {cmd}", evidence="workbench.json")]
        return []

    def similar_change(self) -> list[Fact]:
        hist = self.root / "docs" / "changelog.md"
        if hist.exists() and "dedupe" in hist.read_text(encoding="utf-8").lower():
            return [Fact("changelog 有过类似去重改动", evidence="docs/changelog.md")]
        return []

    def collect(self, *, symbol: str, topic: str) -> list[Fact]:
        """
        Returns:
            facts: 六类约束的并集。
        """
        out: list[Fact] = []
        out.extend(self.callers_of(symbol))
        out.extend(self.nearest_test(topic))
        out.extend(self.public_contract())
        out.extend(self.conventions())
        out.extend(self.verify_command())
        out.extend(self.similar_change())
        return out


class FramingGate:
    """先框任务：验收写完、must_ask 关闭，才能变更。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.path = self.root / "task_frame.json"
        self.impl_started = False

    def save(self, frame: TaskFrame) -> Path:
        self.path.write_text(json.dumps(frame.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        return self.path

    def load(self) -> TaskFrame:
        data = json.loads(self.path.read_text(encoding="utf-8"))
        return TaskFrame(
            goal=data["goal"],
            repo_facts=[Fact(**f) for f in data.get("repo_facts", [])],
            allowed_paths=list(data.get("allowed_paths", [])),
            forbidden_paths=list(data.get("forbidden_paths", [])),
            acceptance=list(data.get("acceptance", [])),
            unknowns=[Unknown(**u) for u in data.get("unknowns", [])],
            assumptions=list(data.get("assumptions", [])),
        )

    def resolve(self, text: str, *, resolution: str) -> TaskFrame:
        """
        Args:
            text: 未知原文。
            resolution: 人给的答案或授权。
        Returns:
            frame: 更新后的框。
        """
        frame = self.load()
        for u in frame.unknowns:
            if u.text == text:
                u.resolved = True
                u.resolution = resolution
        self.save(frame)
        return frame

    def start_impl(self) -> TaskFrame:
        """
        Returns:
            frame: 已通过门禁的框。
        """
        frame = self.load()
        frame.validate()
        self.impl_started = True
        return frame


def make_mail_repo() -> Path:
    """
    Returns:
        root: 「邮件去重」迷你仓库。
    """
    root = Path(tempfile.mkdtemp(prefix="frame_task_"))
    (root / "src" / "email").mkdir(parents=True)
    (root / "tests").mkdir()
    (root / "docs").mkdir()
    (root / "src" / "email" / "send.py").write_text(
        "from src.email.dedupe import seen\n\ndef send(addr: str) -> None:\n    if seen(addr):\n        return\n",
        encoding="utf-8",
    )
    (root / "src" / "email" / "dedupe.py").write_text(
        "def seen(addr: str) -> bool:\n    return False\n",
        encoding="utf-8",
    )
    (root / "tests" / "test_send.py").write_text(
        "# nearest: send path, no dedupe cases yet\nfrom src.email.send import send\n",
        encoding="utf-8",
    )
    (root / "README.md").write_text("Mailer. Repeat sends should be idempotent.\n", encoding="utf-8")
    (root / "AGENTS.md").write_text("Mail changes live under src/email.\n", encoding="utf-8")
    (root / "workbench.json").write_text(json.dumps({"test_command": "python -m pytest -q"}), encoding="utf-8")
    (root / "docs" / "changelog.md").write_text("- 2025: webhook dedupe at ingress\n", encoding="utf-8")
    return root


print("frame the task ready | 6-field frame + recon + must_ask gate")


## 2. 玩具示例

侦查出事实、三种句子分类、验收优先、must_ask 挡实现、人答完才能开改。


In [ ]:
def demo_frame_the_task() -> None:
    """侦查出事实、假设被揭穿、must_ask 挡实现、验收优先。"""
    root = make_mail_repo()
    try:
        recon = Recon(root)
        facts = recon.collect(symbol="seen", topic="send")
        assert any("调用" in f.text for f in facts)
        assert any("test_send" in f.evidence for f in facts)
        assert any("idempotent" in f.text.lower() or "重复" in f.text or "幂等" in f.text or "契约" in f.text for f in facts)
        print("recon constraints ok")

        assert classify_sentence("去重应忽略大小写", evidence=None) == "assumption"
        assert classify_sentence("send.py 调用了 seen", evidence="src/email/send.py") == "fact"
        assert classify_sentence("跨进程是否共享去重表？") == "unknown"
        print("fact vs assumption vs unknown ok")

        frame = TaskFrame(
            goal="给邮件添加重复保护",
            repo_facts=facts,
            allowed_paths=["src/email/**"],
            forbidden_paths=["README.md"],
            acceptance=[],  # 故意先空
            unknowns=[
                Unknown("跨进程是否共享去重表？", "must_ask"),
                Unknown("用内存 set 还是 redis？", "decidable"),
                Unknown("changelog 里 ingress 去重是否可复用？", "discoverable"),
                Unknown("暗主题", "out_of_scope"),
            ],
            assumptions=["去重应忽略大小写"],
        )
        gate = FramingGate(root)
        gate.save(frame)
        try:
            gate.start_impl()
            raise AssertionError("empty acceptance")
        except FrameIncomplete:
            print("acceptance-before-impl ok")

        frame.acceptance = ["连续两次 send(same) 第二次 no-op；pytest -q"]
        gate.save(frame)
        try:
            gate.start_impl()
            raise AssertionError("must_ask still open")
        except MustAskBlocking:
            print("must_ask blocks impl ok")

        gate.resolve("跨进程是否共享去重表？", resolution="本轮单进程内存表，跨进程非目标")
        closed = gate.start_impl()
        assert closed.open_must_ask() == []
        assert gate.impl_started
        print("resolved must_ask then impl ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_frame_the_task()


## 3. 生产级：LangChain 任务框工具 + DeepSeek

工具：`recon_repo` / `draft_frame` / `human_resolve` / `start_impl`。agent 不能自己关 must_ask。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
import threading
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_GATE: FramingGate | None = None
PROD_LOCK = threading.Lock()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_frame_repo() -> str:
    """
    Returns:
        json: 新仓库路径。
    """
    global PROD_ROOT, PROD_GATE
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = make_mail_repo()
    PROD_GATE = FramingGate(PROD_ROOT)
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


def _gate() -> FramingGate:
    if PROD_GATE is None:
        reset_frame_repo()
    assert PROD_GATE is not None
    return PROD_GATE


class EmptyArgs(BaseModel):
    pass


class FrameArgs(BaseModel):
    goal: str
    allowed_paths_csv: str = Field("src/email/**")
    forbidden_paths_csv: str = Field("README.md")
    acceptance_csv: str = Field("", description="分号分隔的验收；空则未写")
    must_ask: str = Field("", description="必须问人的未知，可空")
    assumptions_csv: str = Field("", description="分号分隔，无证据的句子")


class ResolveArgs(BaseModel):
    text: str
    resolution: str
    user_id: str = Field("human")


class SentenceArgs(BaseModel):
    text: str
    evidence: str = ""


def build_frame_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        with PROD_LOCK:
            return reset_frame_repo()

    def _recon(**kwargs: Any) -> str:
        with PROD_LOCK:
            facts = Recon(_gate().root).collect(symbol="seen", topic="send")
            return json.dumps([f.to_dict() for f in facts], ensure_ascii=False)

    def _classify(**kwargs: Any) -> str:
        a = SentenceArgs(**kwargs)
        kind = classify_sentence(a.text, evidence=a.evidence or None)
        return json.dumps({"kind": kind, "text": a.text}, ensure_ascii=False)

    def _draft(**kwargs: Any) -> str:
        with PROD_LOCK:
            a = FrameArgs(**kwargs)
            facts = Recon(_gate().root).collect(symbol="seen", topic="send")
            unknowns: list[Unknown] = []
            if a.must_ask.strip():
                unknowns.append(Unknown(a.must_ask.strip(), "must_ask"))
            acc = [x.strip() for x in a.acceptance_csv.split(";") if x.strip()]
            assumptions = [x.strip() for x in a.assumptions_csv.split(";") if x.strip()]
            frame = TaskFrame(
                goal=a.goal,
                repo_facts=facts,
                allowed_paths=[x.strip() for x in a.allowed_paths_csv.split(",") if x.strip()],
                forbidden_paths=[x.strip() for x in a.forbidden_paths_csv.split(",") if x.strip()],
                acceptance=acc,
                unknowns=unknowns,
                assumptions=assumptions,
            )
            _gate().save(frame)
            return json.dumps(frame.to_dict(), ensure_ascii=False)

    def _read(**kwargs: Any) -> str:
        with PROD_LOCK:
            p = _gate().path
            if not p.exists():
                return json.dumps({"ok": False, "error": "task_frame.json missing"}, ensure_ascii=False)
            return p.read_text(encoding="utf-8")

    def _ask(**kwargs: Any) -> str:
        with PROD_LOCK:
            a = ResolveArgs(**kwargs)
            if a.user_id.lower() in {"agent", "assistant"}:
                return json.dumps({"ok": False, "error": "agent cannot close must_ask"}, ensure_ascii=False)
            frame = _gate().resolve(a.text, resolution=a.resolution)
            return json.dumps(frame.to_dict(), ensure_ascii=False)

    def _impl(**kwargs: Any) -> str:
        with PROD_LOCK:
            try:
                frame = _gate().start_impl()
                return json.dumps({"ok": True, "started": True, "goal": frame.goal}, ensure_ascii=False)
            except (FrameIncomplete, MustAskBlocking) as e:
                return json.dumps({"ok": False, "started": False, "error": str(e)}, ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_repo", description="重建邮件去重演示仓库。", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="recon_repo", description="侦查约束（调用方/测试/契约/规矩/验证/类似改动）。", func=_recon, args_schema=EmptyArgs),
        StructuredTool.from_function(name="classify_sentence", description="事实/假设/未知。", func=_classify, args_schema=SentenceArgs),
        StructuredTool.from_function(name="draft_frame", description="写 task_frame.json（六字段）。", func=_draft, args_schema=FrameArgs),
        StructuredTool.from_function(name="read_frame", description="读任务框。", func=_read, args_schema=EmptyArgs),
        StructuredTool.from_function(name="human_resolve", description="人类关闭 must_ask。", func=_ask, args_schema=ResolveArgs),
        StructuredTool.from_function(name="start_impl", description="框闭合且无 must_ask 才能开始改代码。", func=_impl, args_schema=EmptyArgs),
    ]


FRAME_TOOLS = build_frame_tools()


def build_frame_agent():
    """
    Returns:
        agent: 先框任务再实现的 control agent。
    """
    system = (
        "你是任务框代理。先 recon_repo，再 draft_frame。"
        "必须先写验收。must_ask 未关闭时不要 start_impl，也不要自己 human_resolve。\n"
        "无证据的句子标成 assumptions。用中文短说明。"
    )
    return create_agent(get_llm(), FRAME_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"frame the task production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证 must_ask 门禁。


In [ ]:
def demo_prod_frame() -> None:
    """脚本门禁 +（有 key 时）DeepSeek 先框后改。"""
    reset_frame_repo()
    facts = Recon(_gate().root).collect(symbol="seen", topic="send")
    frame = TaskFrame(
        goal="给邮件添加重复保护",
        repo_facts=facts,
        allowed_paths=["src/email/**"],
        forbidden_paths=["README.md"],
        acceptance=["两次 send 同地址，第二次 no-op"],
        unknowns=[Unknown("跨进程共享？", "must_ask")],
    )
    _gate().save(frame)
    try:
        _gate().start_impl()
        raise AssertionError("must_ask")
    except MustAskBlocking:
        pass
    _gate().resolve("跨进程共享？", resolution="单进程即可")
    _gate().start_impl()
    print("=== scripted ===")
    print(json.dumps({"blocked_then_started": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_frame_agent()
    prompt = (
        "reset_repo，recon_repo，classify_sentence 文本=去重应忽略大小写（无证据），"
        "draft_frame goal=给邮件添加重复保护 acceptance 先空 must_ask=跨进程是否共享去重表？，"
        "尝试 start_impl（应失败），补 draft_frame 带验收但仍保留 must_ask，再 start_impl（仍应失败），"
        "不要自己 human_resolve；中文说明为何先框再改。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    # agent 被要求不要 resolve，最终不应已经 start
    # 但第二次 start 仍应失败；若它补了验收却没关 must_ask
    started = _gate().impl_started
    frame = _gate().load() if _gate().path.exists() else None
    if started:
        # 只有在 must_ask 已关时才合法
        assert frame is None or not frame.open_must_ask()
    else:
        assert frame is None or frame.open_must_ask() or not frame.acceptance
    print("PROD DEMO OK")


demo_prod_frame()
